### https://www.kaggle.com/competitions/drawing-with-llms

In [1]:
import kagglehub
import pandas as pd


In [2]:
# from unsloth import FastLanguageModel
# import torch

# max_seq_length = 2048 
# dtype = ( None )
# load_in_4bit = False 


# model, tokenizer = FastLanguageModel.from_pretrained(
#     model_name="./lora/lora_16bit_merged/",
#     max_seq_length=max_seq_length,
#     dtype=dtype,
#     load_in_4bit=load_in_4bit,
# )
# FastLanguageModel.for_inference(model) # Enable native 2x faster inference

In [3]:
# # I highly do NOT suggest - use Unsloth if possible
# from peft import AutoPeftModelForCausalLM
# from transformers import AutoTokenizer
# model = AutoPeftModelForCausalLM.from_pretrained(
#     "lora_model", # YOUR MODEL YOU USED FOR TRAINING
#     load_in_4bit = False,
# )
# tokenizer = AutoTokenizer.from_pretrained("lora_model")

In [4]:
#| export
import concurrent
import io
import logging
import re
import re2

import cairosvg
import kagglehub
import torch
from lxml import etree
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

svg_constraints = kagglehub.package_import('metric/svg-constraints')

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print('DEVICE',DEVICE)

class Model:
    def __init__(self):
        # Quantization Configuration
        max_seq_length = 2048  
        torch_dtype = torch.float16  # Adjust dtype based on your model precision
        
        #self.model_path = kagglehub.model_download('vinothkumarsekar89/llama_3p2_1b_svg_16bit_merged/transformers/01')
        self.model_path = './lora/lora_16bit_merged/' 
        
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_path)
        self.model = AutoModelForCausalLM.from_pretrained(
            self.model_path,
            torch_dtype=torch_dtype,
            device_map="auto"
        )
        self.model.eval()
        
        self.default_svg = """<svg width="256" height="256" viewBox="0 0 256 256"><circle cx="50" cy="50" r="40" fill="red" /></svg>"""
        self.constraints = svg_constraints.SVGConstraints()
        self.timeout_seconds = 90

    
    def predict(self, description: str, max_new_tokens=1024) -> str:
        default_svg = self.default_svg

        def clean_and_extract_svgs(text):
            cleaned_svgs = []
           
            # Remove any text before the first <svg>
            text = re.sub(r'^.*?(<svg\b)', r'\1', text, flags=re.DOTALL)
        
            # Find all <svg> tags (including nested)
            svg_blocks = re.findall(r'<svg\b.*?</svg>', text, re.DOTALL)
        
            if svg_blocks:
                # If there are multiple <svg> blocks, we pick the innermost one (which is the last in the list)
                tmp = re.findall(r'<svg\b.*?', svg_blocks[-1], re.DOTALL)
                if len(tmp) > 1:
                    tmp2 = svg_blocks[-1].split('<svg')
                    cleaned_svgs.append('<svg ' + tmp2[-1])
                else:
                    cleaned_svgs.append(svg_blocks[-1])
            else:
                # Handle incomplete SVGs
                if "<svg" in text and "</svg>" not in text:
                    # Remove any text before <svg> (again, in case it was incomplete)
                    text = re.sub(r'^.*?(<svg\b)', r'\1', text, flags=re.DOTALL)
                    # Ensure it ends with a proper closing </svg>
                    text += "</svg>"
                    cleaned_svgs.append(text)
                else:
                    cleaned_svgs.append(default_svg)  # No valid SVG found

            # Removes the last incomplete element, like <incomp....</svg>
            return cleaned_svgs[0].rsplit('\n', 1)[0] + '\n   </svg>'

        # Clean unconvertible SVG with default SVG
        def svg_conversion_check(topic, base_svg_code):
            try:
                # Try to convert the SVG to PNG
                cairosvg.svg2png(bytestring=base_svg_code.encode('utf-8'), write_to="temp.png")
                return base_svg_code
                
            except Exception as e:
                # If conversion fails, return the default SVG
                print(f"Failed to convert {topic} due to {str(e)}\n The base code:\n{}\nReturning default SVG.")
                return default_svg
            
        def get_response():
            # Define the Alpaca prompt template
            alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.
        
            ### Instruction:
            Please write an SVG code for the given topic?
        
            ### Input:
            {}
        
            ### Response:
            """
        
            # Format the input text properly for the prompt
            formatted_input = alpaca_prompt.format(description)
        
            # Tokenize the formatted input
            inputs = self.tokenizer([formatted_input], return_tensors="pt").to(DEVICE)
        
            # Generate the response from the model
            outputs = self.model.generate(**inputs, max_new_tokens=2048, use_cache=True)
            
            output_decoded = self.tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
          
            return output_decoded

        output_decoded = get_response()
        base_svg_code = clean_and_extract_svgs(output_decoded)
        final_svg_code = svg_conversion_check(description, base_svg_code)
        
        return final_svg_code



This code could modify your python environment or operating system.

Review this code at https://www.kaggle.com/code/metric/svg-constraints/versions/1
or in your download cache at /home/vino/.cache/kagglehub/notebooks/metric/svg-constraints/output/versions/1

It is strongly recommended that you run this code within a container
such as Docker to provide a secure, isolated execution environment.
See https://www.kaggle.com/docs/packages for more information.

Do you want to proceed? (y)es/[no]:  y


DEVICE cuda


In [5]:
model=Model()

In [6]:
model.predict('sun rising in the east')

Device: cuda
Model is on cuda:0


'<svg viewBox="0 0 200 200" width="200" height="200" xmlns="http://www.w3.org/2000/svg">\n               <!-- Sky -->\n               <rect x="0" y="0" width="200" height="100" fill="lightblue" />\n               <!-- Sun -->\n               <circle cx="100" cy="100" r="30" fill="yellow" opacity="0.8" />\n               <!-- Sun Rays -->\n               <g stroke="yellow" stroke-width="2" opacity="0.5">\n                  <line x1="100" y1="80" x2="100" y2="20" />\n                  <line x1="120" y1="90" x2="140" y2="90" />\n                  <line x1="140" y1="70" x2="160" y2="70" />\n                  <line x1="160" y1="60" x2="180" y2="60" />\n                  <line x1="180" y1="50" x2="200" y2="50" />\n                  <line x1="200" y1="30" x2="200" y2="10" />\n               </g>\n   </svg>'

In [7]:
from transformers import AutoProcessor, AutoModel
model_sl = AutoModel.from_pretrained("google/siglip-so400m-patch14-384")
processor_sl = AutoProcessor.from_pretrained("google/siglip-so400m-patch14-384")

In [8]:
import torch
from PIL import Image
import cairosvg
import os

def svgMetric(prompt, svg):
    try:
        # Convert SVG to PNG
        cairosvg.svg2png(svg, write_to="./tmp/temp.png")
        
        # Open and process the image
        image = Image.open('./tmp/temp.png').convert("RGB")
        texts = ["SVG illustration of " + prompt]
        inputs = processor_sl(text=texts, images=image, padding="max_length", return_tensors="pt")
        
        # Inference without gradient tracking
        with torch.no_grad():
            outputs = model_sl(**inputs)
        
        logits_per_image = outputs.logits_per_image
        probs = torch.sigmoid(logits_per_image)
        
        # Clean up temporary PNG file
        #os.remove('./tmp/temp.png')
        
        return probs[0][0].item()

    
    except Exception as e:
        print(f"An error occurred: {e}")
        return None

In [9]:
import pandas as pd
df=pd.read_csv('svg_score_test.csv')
df=df[df['score'] > 0.6]
df=df[['topic','svg_code']]

In [11]:
from tqdm import tqdm
tqdm.pandas()
df['base_svg_code'] = df['topic'].progress_apply(lambda x: model.predict(x))

  3%|█▏                                          | 2/74 [00:04<02:25,  2.02s/it]

Device: cuda
Model is on cuda:0


  4%|█▊                                          | 3/74 [00:08<03:32,  2.99s/it]

Device: cuda
Model is on cuda:0


  5%|██▍                                         | 4/74 [00:11<03:29,  3.00s/it]

...

Device: cuda
Model is on cuda:0


100%|███████████████████████████████████████████| 74/74 [08:01<00:00,  7.01s/it]

Device: cuda
Model is on cuda:0


100%|███████████████████████████████████████████| 74/74 [08:06<00:00,  6.57s/it]

Device: cuda
Model is on cuda:0


In [12]:
# Using apply to process each row in the DataFrame
df['base_score'] = df.progress_apply(lambda row: svgMetric(row['topic'], row['base_svg_code']), axis=1)

100%|███████████████████████████████████████████| 74/74 [02:02<00:00,  1.65s/it]


In [13]:
df['base_score'].mean()

np.float64(0.3623888401356792)